<div style="width: 30em; float: right; padding: 3em; border: 5px red solid; background-color: darkred; color: white"><p style="font-size: large; font-weight:bold">Rename this notebook before running any cells!</p><ol><li>Right-click on the tab title above or on the notebook in the file-browser on the left and select rename.</li><li>Remove the "_orig" part of the file name.</li></ol><p style="font-size: large; font-weight:bold">Check the notebook kernel!</p><ol><li>Check the current notebook kernel in the upper right corner.</li><li>Set to "Python 3.12 (Conda)".</li></ol></div>

# Session 5 — Reading and Writing Files: Text, CSV, JSON, and JSONL

IFI 8410 · Module 1

A notebook loses its state when the kernel restarts; a file outlives the session
that created it. That is what allows one program's output to become another
program's input, and it is why almost every data-science workflow begins and ends
with a file:

> **Input files → processing program → output files**

Until now every dataset in this course has been typed straight into a cell: the
coffee-cart `sales` list in Sessions 3 and 4, the bike-share trips in HW02, the
scholarship applications in HW03. This notebook moves that same coffee-cart data
out of the notebook and onto disk — as plain text, as CSV, as JSON, and as JSON
Lines — and reads it back again.

By the end you should be able to:

- explain how the **current working directory** decides where a relative path points
- build paths with `pathlib.Path` and diagnose a `FileNotFoundError`
- open, read, and write text files safely with `with open(...)`, and choose a file mode
- read and write CSV with `csv.DictReader` and `csv.DictWriter`, and explain why
  `line.split(",")` is not a CSV parser
- convert between Python values and JSON with `json.load`/`json.dump` and
  `json.loads`/`json.dumps`
- read and write **JSON Lines** one record at a time, and report bad lines by number
- choose between CSV, JSON, and JSONL for a given dataset, and say why
- validate an input file before trusting the answer computed from it

**How to work through it.** Run every code cell in order, top to bottom. Later cells
use files and variables created by earlier ones. Where you see a **Try it** cell,
predict the result before you run it.

The companion notebook, `Python_Modules_and_Scripts_orig.ipynb`, turns the
processing code from this notebook into a program you can run from a terminal.

---

## 0. Setup: a workspace for this notebook

This notebook creates files. So that it never touches anything else, everything it
writes goes into one folder, `files_workspace/`, next to the notebook.

The cell below **deletes and recreates** that folder every time it runs. That is
deliberate: re-running the notebook starts from a clean state, which is exactly the
property this session is about. It also means that `files_workspace/` is not a place
to keep your own work. Put that in `Student-Notes/`, which is yours.

In [ ]:
import shutil
from pathlib import Path

WORKSPACE = Path("files_workspace")

# Only ever delete the folder this notebook owns.
if WORKSPACE.name == "files_workspace" and WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

(WORKSPACE / "data" / "raw").mkdir(parents=True)
(WORKSPACE / "data" / "processed").mkdir(parents=True)
(WORKSPACE / "output").mkdir(parents=True)

print("Workspace ready:", WORKSPACE.resolve())
for folder in sorted(WORKSPACE.rglob("*")):
    print("  ", folder)

We also need the dataset. This is the Session 3 coffee-cart table, unchanged: a
**list of dictionaries**, one dictionary per transaction, keys as column names.

In [ ]:
sales = [
    {"id": 101, "item": "coffee", "category": "drink", "price": 3.50, "day": "Mon", "hour": 8,  "customer_type": "student"},
    {"id": 102, "item": "tea",    "category": "drink", "price": 2.75, "day": "Mon", "hour": 9,  "customer_type": "faculty"},
    {"id": 103, "item": "muffin", "category": "food",  "price": 2.50, "day": "Mon", "hour": 9,  "customer_type": "student"},
    {"id": 104, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "hour": 8,  "customer_type": "student"},
    {"id": 105, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Tue", "hour": 10, "customer_type": "staff"},
    {"id": 106, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "hour": 10, "customer_type": "faculty"},
    {"id": 107, "item": "tea",    "category": "drink", "price": 2.75, "day": "Wed", "hour": 8,  "customer_type": "student"},
    {"id": 108, "item": "cookie", "category": "food",  "price": 1.75, "day": "Wed", "hour": 11, "customer_type": "student"},
    {"id": 109, "item": "coffee", "category": "drink", "price": 3.50, "day": "Wed", "hour": 11, "customer_type": "staff"},
    {"id": 110, "item": "muffin", "category": "food",  "price": 2.50, "day": "Thu", "hour": 9,  "customer_type": "faculty"},
    {"id": 111, "item": "coffee", "category": "drink", "price": 3.50, "day": "Thu", "hour": 9,  "customer_type": "student"},
    {"id": 112, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Fri", "hour": 8,  "customer_type": "student"},
]

print(f"{len(sales)} transactions, columns: {list(sales[0])}")

---

## 1. Why files matter

Everything in the `sales` list above lives in the kernel's memory. Restart the
kernel and it is gone; the only reason you can get it back is that the *code that
builds it* is saved in the notebook.

Notebook state has other hidden dependencies, too:

- Variables disappear when the kernel restarts.
- Cells may have been run out of order, or run twice.
- An earlier cell may have changed data that a later cell silently depends on.
- Which input file was used, and which output files were written, may not be
  visible anywhere.
- Another person may be unable to recreate your result from a clean start.

A file fixes the first problem outright and makes the others visible. A workflow
that reads named input files and writes named output files is a **defined
transformation**:

```text
sales.csv ──▶ summarize_sales.py ──▶ item_revenue.csv
```

Such a workflow can be re-run next week or by a teammate, reviewed and tested
outside a notebook, archived together with its inputs and outputs, and eventually
automated. That is the beginning of **reproducible data science**.

### The project-management view

Many of you will manage analytics projects rather than write every line. The
questions a file-based workflow lets you ask — and demand answers to — are:

- What files are the inputs, and who provides them?
- What format, columns, units, dates, identifiers, and encoding are expected?
- What happens when an input is missing or malformed?
- What outputs are produced, and how are they checked?
- Can someone other than the author rerun it without their laptop?

Keep these in mind; every section below answers one of them in code.

---

## 2. Files, folders, and paths

A **file path** tells the operating system where a file is. You have already met
paths in HW01, where your program read newsgroup messages out of a directory tree.

```text
data/raw/sales.csv                          relative
/home/student/ifi8410/data/raw/sales.csv    absolute (Linux, macOS)
C:\Users\Student\ifi8410\data\raw\sales.csv  absolute (Windows)
```

### Absolute versus relative

An **absolute path** starts at the root of the file system. It is unambiguous — on
one computer. Hard-coded into shared code, it breaks for everyone whose username or
folder layout is different:

```python
open("/Users/amina/Desktop/IFI8410/data/sales.csv")   # works on exactly one laptop
```

A **relative path** is interpreted starting from the **current working directory**.
`data/raw/sales.csv` means: *from where I am now, go into `data`, then `raw`, then
open `sales.csv`.* Relative paths make a project portable, provided everyone runs it
from the same starting folder.

### The current working directory

The working directory is not a property of the file you are reading; it is a
property of the *running process*. Ask Python where it is:

In [ ]:
from pathlib import Path

print("Working directory:", Path.cwd())

In JupyterLab, a notebook's working directory is **the folder that contains the
notebook**. In a terminal, it is whatever folder you last `cd`-ed into. The same
relative path can therefore point at two different files — or at nothing — depending
on where the code runs. That single fact explains most of the `FileNotFoundError`s
you will ever see.

List what is visible from here:

In [ ]:
for item in sorted(Path.cwd().iterdir()):
    kind = "dir " if item.is_dir() else "file"
    print(kind, item.name)

### Build paths with `pathlib`

`pathlib.Path` represents a path as an object instead of a string. The `/` operator
joins parts with the correct separator for the operating system, and the object knows
how to answer questions about itself.

In [ ]:
input_path = WORKSPACE / "data" / "raw" / "sales.csv"

print("path     :", input_path)
print("name     :", input_path.name)
print("stem     :", input_path.stem)
print("suffix   :", input_path.suffix)
print("parent   :", input_path.parent)
print("absolute :", input_path.resolve())
print("exists?  :", input_path.exists())      # not yet -- we have not written it

`input_path.resolve()` turns a relative path into the absolute path it refers to
*from the current working directory*. When a program cannot find a file, printing the
resolved path is the fastest way to see where it actually looked.

### The common `FileNotFoundError`

Here is the error, caught so that the notebook can keep running:

In [ ]:
try:
    with open(input_path, encoding="utf-8") as file:
        contents = file.read()
except FileNotFoundError as error:
    print(f"{type(error).__name__}: {error}")

At this stage of the course a `FileNotFoundError` is almost always a **path**
problem, not a logic problem. Check, in order:

1. Is the filename spelled exactly right?
2. Does the capitalization match? On Linux — the course server — `Sales.csv` and
   `sales.csv` are different files.
3. Is the extension right: `.csv` versus `.CSV` versus `.txt`?
4. Does each folder in the path exist *under the working directory*?
5. Is Python running from the folder you think it is? (`Path.cwd()`)
6. Is a notebook using a different working directory than your terminal?
7. Is the file actually in Downloads, or somewhere other than the project?

A good program checks before it opens, and says *where* it looked:

In [ ]:
def require_file(path: Path) -> Path:
    """Return path unchanged, or raise FileNotFoundError naming where Python looked."""
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path.resolve()}")
    return path


try:
    require_file(input_path)
except FileNotFoundError as error:
    print(error)

`require_file` follows the Session 4 habits: it has one job, it returns rather than
prints, and its error message is written for the person who has to fix the problem.

In [ ]:
### Try it: build a Path to "files_workspace/output/summary.txt" using the / operator.
### Print its name, suffix, and parent, and whether it exists. Then call
### require_file on Path("Sales.csv") and read the error message carefully.

### Enter your code here ###

---

## 3. Reading and writing text files

A plain-text file is a sequence of characters, usually organized into lines. `.txt`,
`.csv`, `.json`, `.jsonl`, `.log`, `.md`, and `.py` are all text files; the extension
is only a hint about how the text is organized.

### Write a text file

The standard pattern is a `with` statement around `open(path, mode, encoding)`:

In [ ]:
notes_path = WORKSPACE / "data" / "raw" / "shift_notes.txt"

with open(notes_path, mode="w", encoding="utf-8") as file:
    file.write("Coffee cart shift notes\n")
    file.write("Mon: espresso machine slow before 9am\n")
    file.write("\n")
    file.write("Tue: ran out of bagels at 10am\n")
    file.write("Wed: new cookie supplier, Café Zoë\n")
    file.write("   \n")
    file.write("Thu: card reader offline for 20 minutes\n")

print("Wrote", notes_path, "-", notes_path.stat().st_size, "bytes")

Three details:

- `file.write()` does **not** add a newline. You write `"\n"` yourself.
- `mode="w"` creates the file, or **replaces** it if it already exists.
- `encoding="utf-8"` is explicit. `Café Zoë` contains non-ASCII characters; without
  a stated encoding, what gets written depends on the machine's default settings.

### Read the whole file

In [ ]:
with open(notes_path, mode="r", encoding="utf-8") as file:
    contents = file.read()

print(contents)
print("characters:", len(contents))

### Why `with`?

`with` is a **context manager**: when the indented block ends — normally, or because
of an error — the file is closed. You can see this by checking `file.closed`:

In [ ]:
with open(notes_path, encoding="utf-8") as file:
    print("inside the block, closed? ", file.closed)

print("after the block, closed?  ", file.closed)

Closing by hand works only if nothing goes wrong first. In the cell below, the error
happens before `close()`, so the file is left open:

In [ ]:
file = open(notes_path, encoding="utf-8")
try:
    contents = file.read()
    result = 10 / 0                  # something fails part-way through
    file.close()                     # never reached
except ZeroDivisionError:
    print("manual close -> still open?", not file.closed)
file.close()                         # tidy up the demonstration

try:
    with open(notes_path, encoding="utf-8") as file:
        contents = file.read()
        result = 10 / 0
except ZeroDivisionError:
    print("with block   -> still open?", not file.closed)

An open file holds an operating-system resource, and on some systems it locks the
file against other programs. A script that processes thousands of files and forgets
to close them eventually fails with *too many open files*. `with` makes that
impossible to forget.

### File modes

| Mode | Meaning | Typical use |
|---|---|---|
| `"r"` | read an existing file (the default) | reading input data |
| `"w"` | write, **replacing** any existing contents | a fresh output file |
| `"a"` | append to the end, creating the file if needed | logs, growing record files |
| `"x"` | create, **failing** if the file already exists | refusing to overwrite |
| `"rb"`, `"wb"` | read or write raw bytes | images, PDFs, compressed data |

`"w"` is the dangerous one: it empties the file the moment it is opened, before you
write a single character.

In [ ]:
scratch_path = WORKSPACE / "output" / "mode_demo.txt"

with open(scratch_path, "w", encoding="utf-8") as file:
    file.write("first version\n")

with open(scratch_path, "w", encoding="utf-8") as file:     # replaces
    file.write("second version\n")

with open(scratch_path, "a", encoding="utf-8") as file:     # adds to the end
    file.write("appended line\n")

print(scratch_path.read_text(encoding="utf-8"))

try:
    with open(scratch_path, "x", encoding="utf-8") as file:  # refuses
        file.write("this never happens\n")
except FileExistsError as error:
    print(f"mode 'x' -> {type(error).__name__}: {error}")

`Path` objects also have one-line shortcuts — `path.read_text()` and
`path.write_text()` — that open, read or write, and close in a single call. They are
convenient for small files. The `with open(...)` form is the one to learn first,
because it is what you need for reading line by line and for CSV and JSON.

### Encoding: characters versus bytes

A file on disk stores **bytes**. An encoding is the rule that turns characters into
bytes and back. In UTF-8, plain English letters take one byte each, but `é` and `ë`
take two:

In [ ]:
line = "Café Zoë"
print("characters:", len(line))
print("bytes     :", len(line.encode("utf-8")), line.encode("utf-8"))

# Reading with the wrong encoding does not always fail -- sometimes it just garbles.
raw_bytes = notes_path.read_bytes()
print()
print("read as utf-8  :", raw_bytes.decode("utf-8").splitlines()[4])
print("read as latin-1:", raw_bytes.decode("latin-1").splitlines()[4])

The second line is *mojibake*: no error, just wrong text. That is why the course uses
`encoding="utf-8"` on every `open()` of a text file unless a data provider tells you
otherwise.

### Read line by line

`file.read()` loads everything at once. Iterating over the file object instead gives
one line at a time, which works for a file of any size. Use `repr()` to see what a
line really contains:

In [ ]:
with open(notes_path, encoding="utf-8") as file:
    for line in file:
        print(repr(line))

Every line ends in `"\n"`, and `print()` adds another, which is why printing raw lines
double-spaces them. Strip deliberately:

- `line.rstrip("\n")` removes only the line ending.
- `line.rstrip()` removes the line ending and any trailing spaces.
- `line.strip()` removes whitespace from **both** ends.

None of these is "the right cleaning". Leading spaces can be meaningful — indentation
in a `.py` file, for example. Pick the one that matches what the data means.

In [ ]:
with open(notes_path, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number:>2} | {line.rstrip()!r}")

### A counting pattern you already know

Session 3's accumulator pattern works on a file exactly as it worked on a list. Here
we count non-empty lines, and collect the days mentioned:

In [ ]:
def summarize_notes(path: Path) -> dict:
    """Return counts of total, blank, and non-blank lines, and the days mentioned."""
    total = blank = 0
    days = []

    with open(path, encoding="utf-8") as file:
        for line in file:
            total += 1
            text = line.strip()
            if not text:
                blank += 1
                continue
            day, separator, _ = text.partition(":")
            if separator:
                days.append(day)

    return {"lines": total, "blank": blank, "nonblank": total - blank, "days": days}


print(summarize_notes(notes_path))

The line `"   \n"` counts as blank, because `.strip()` reduces it to `""`, and an empty
string is falsy. That is a decision, and a program should make it on purpose.

### Handle a header line

Structured text usually starts with a header. `next(file)` takes the next line from
the file iterator; the `for` loop then continues from the line after it. Passing a
default — `next(file, None)` — avoids a `StopIteration` crash on an empty file.

In [ ]:
empty_path = WORKSPACE / "data" / "raw" / "empty.txt"
empty_path.write_text("", encoding="utf-8")


def read_with_header(path: Path) -> tuple[str | None, list[str]]:
    """Return (header, body lines). The header is None for an empty file."""
    with open(path, encoding="utf-8") as file:
        header = next(file, None)
        if header is None:
            return None, []
        return header.rstrip(), [line.rstrip() for line in file]


header, body = read_with_header(notes_path)
print("header:", header)
print("body  :", body)

print(read_with_header(empty_path))

### Verify by reading back

A write that raised no error is not the same as a correct output. The habit worth
building is:

> **write → read it back → check it against what you expected**

In [ ]:
report_path = WORKSPACE / "output" / "notes_summary.txt"
summary = summarize_notes(notes_path)

with open(report_path, "w", encoding="utf-8") as file:
    file.write("Shift Notes Summary\n")
    file.write("===================\n")
    file.write(f"Input file: {notes_path}\n")
    file.write(f"Lines read: {summary['lines']}\n")
    file.write(f"Blank lines: {summary['blank']}\n")
    file.write(f"Days mentioned: {', '.join(summary['days'])}\n")

written = report_path.read_text(encoding="utf-8")
print(written)

assert written.startswith("Shift Notes Summary\n"), "header missing"
assert f"Lines read: {summary['lines']}" in written, "line count missing"
print("read-back check passed")

In [ ]:
### Try it: append a line "Fri: record sales day" to shift_notes.txt using mode "a".
### Predict what summarize_notes() returns now, then run it to check.
### What happens if you use mode "w" by mistake? (Re-run the setup cells to recover.)

### Enter your code here ###

---

## 4. CSV files

**CSV** — comma-separated values — is the most common way to move a table between
programs. One row per line, one field per column, a delimiter between fields, and
usually a header row naming the columns.

### Write the coffee-cart table as CSV

`csv.DictWriter` writes a list of dictionaries — the exact shape of `sales`. Its
`fieldnames` argument is the **output schema**: which columns, in which order.

In [ ]:
import csv

sales_csv = WORKSPACE / "data" / "raw" / "sales.csv"
fieldnames = ["id", "item", "category", "price", "day", "hour", "customer_type"]

with open(sales_csv, mode="w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sales)

print(sales_csv.read_text(encoding="utf-8"))

Open `files_workspace/data/raw/sales.csv` in JupyterLab's file browser: it is ordinary
text, and it would open in a spreadsheet too.

**`newline=""`.** The `csv` module handles line endings itself (the CSV standard uses
`\r\n`). Passing `newline=""` to `open()` stops Python from translating them a second
time. Without it, files written on Windows gain blank rows between records. Use it on
every `open()` that feeds `csv`, for reading and writing.

### CSV is a convention, not a complete data model

CSV leaves a lot unspecified: quoting, embedded commas, embedded newlines, missing
fields, other delimiters, encodings, and above all **types** — every value is text.
The menu below has descriptions that contain commas, so the `csv` module quotes them:

In [ ]:
menu = [
    {"item": "coffee", "description": "Drip coffee, 12 oz", "price": "3.50"},
    {"item": "bagel",  "description": "Plain, sesame, or everything", "price": "3.00"},
    {"item": "muffin", "description": 'Blueberry "jumbo" muffin', "price": "2.50"},
]

menu_csv = WORKSPACE / "data" / "raw" / "menu.csv"
with open(menu_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["item", "description", "price"])
    writer.writeheader()
    writer.writerows(menu)

print(menu_csv.read_text(encoding="utf-8"))

Notice the quoting: a field containing a comma is wrapped in `"..."`, and a literal
quote inside a quoted field is doubled (`""jumbo""`). Now compare a hand-rolled parser
with the real one:

In [ ]:
with open(menu_csv, encoding="utf-8", newline="") as file:
    lines = file.read().splitlines()

print("split(','):")
for line in lines[1:]:
    fields = line.split(",")
    print(f"  {len(fields)} fields -> {fields}")

print()
print("csv.reader:")
with open(menu_csv, encoding="utf-8", newline="") as file:
    for row in list(csv.reader(file))[1:]:
        print(f"  {len(row)} fields -> {row}")

`split(",")` cannot tell a comma that separates fields from a comma that is part of
the data, so rows come out with the wrong number of columns — and a price column that
is sometimes a description. The `csv` module gets it right. **Never split CSV lines on
commas by hand.**

### Read with `csv.reader`

`csv.reader` yields each row as a **list of strings**, header included:

In [ ]:
with open(sales_csv, encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)
    print("columns:", header)
    for row in reader:
        print(row)

Positional access — `row[3]` for price — works, but it is unreadable, and it breaks
silently the day someone inserts a column. That is the same argument Session 3 made
for dictionaries over lists: `sale["price"]` means something; `sale[3]` does not.

### Prefer `csv.DictReader`

`csv.DictReader` uses the header row as keys, so each row comes back as a dictionary —
the same list-of-dictionaries shape as `sales`:

In [ ]:
with open(sales_csv, encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

print(len(rows), "rows")
print(rows[0])

for row in rows[:3]:
    print(row["item"], row["price"], row["day"])

### Everything comes back as a string

Compare the first row with the original `sales[0]`:

In [ ]:
print("original :", sales[0])
print("from CSV :", rows[0])
print()
print("types    :", {key: type(value).__name__ for key, value in rows[0].items()})
print("equal?   :", rows[0] == sales[0])

The round trip lost every type. `101` became `"101"` and `3.5` became `"3.5"`. This is
not a bug in `csv`; CSV simply has no way to record types. The consequences are the
string surprises from Session 3:

In [ ]:
prices = [row["price"] for row in rows]
hours = [row["hour"] for row in rows]

print("latest hour    :", max(hours))                        # "9", although 11 is in the data
print("sorted hours   :", sorted(hours))                     # "10" sorts before "8"
print("'3.5' + '2.75' :", prices[0] + prices[1])             # concatenation

try:
    sum(prices)
except TypeError as error:
    print("sum(prices)    :", f"TypeError: {error}")

`max` said the latest sale was at hour `"9"`, and `sorted` put `"10"` and `"11"` before
`"8"`, because strings compare character by character and `"1"` comes before `"8"`. No
error, wrong answer — the worst kind.

**Convert types explicitly, right after reading.** Session 4's `parse_price` (Example
15) was written for exactly this job: it returns a clean `float`, or `None` for a value
that cannot be used, so one bad row does not stop the batch.

In [ ]:
def parse_price(text: object) -> float | None:
    """Return text converted to a price, or None when it cannot be used."""
    try:
        value = float(str(text).strip().lstrip("$"))
    except (TypeError, ValueError):
        return None
    if value < 0:
        return None
    return round(value, 2)


def convert_sale(row: dict) -> dict:
    """Return a new sale dictionary with numeric fields converted from text."""
    return {
        **row,
        "id": int(row["id"]),
        "price": parse_price(row["price"]),
        "hour": int(row["hour"]),
    }


converted = [convert_sale(row) for row in rows]
print(converted[0])
print("matches the original table?", converted == sales)

`{**row, "id": ...}` builds a **new** dictionary: copy every key from `row`, then
override three of them. The rows read from the file are left untouched — Session 4's
"return new data rather than overwrite" habit.

### Validate the columns before trusting the rows

A program that expects a `price` column should say so, and check, before it computes
anything. The set of required columns is a **data contract** between whoever produces
the file and the program that consumes it.

In [ ]:
REQUIRED_COLUMNS = {"id", "item", "category", "price", "day", "hour", "customer_type"}


def read_sales_csv(path: Path) -> list[dict]:
    """Return the rows of a sales CSV as dictionaries of strings.

    Raises:
        FileNotFoundError: if path does not exist.
        ValueError: if the header lacks any required column.
    """
    require_file(path)
    with open(path, encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")
        return list(reader)


print(len(read_sales_csv(sales_csv)), "rows passed the column check")

try:
    read_sales_csv(menu_csv)          # a real CSV -- just not a sales file
except ValueError as error:
    print("ValueError:", error)

`reader.fieldnames or []` guards the empty-file case: for a file with no header at all,
`fieldnames` is `None`, and `set(None)` would raise a confusing `TypeError` instead of
the clear message above.

### Real files are messy

The file below is what a coffee-cart export looks like after a few weeks of real use:
a price typed with a dollar sign, a blank item, a price of `"free"`, a negative price
from a refund, and a row with a stray extra field.

In [ ]:
messy_csv = WORKSPACE / "data" / "raw" / "sales_messy.csv"
messy_csv.write_text(
    "id,item,category,price,day,hour,customer_type\n"
    "201,coffee,drink,3.50,Mon,8,student\n"
    "202,latte,drink,$4.25,Mon,9,faculty\n"
    "203,,food,2.50,Mon,9,student\n"
    "204,cookie,food,free,Tue,10,staff\n"
    "205,tea,drink,-2.75,Tue,10,student\n"
    "206,bagel,food,3.00,Wed,8,student,EXTRA\n"
    "207,\"muffin, blueberry\",food,2.50,Wed,11,faculty\n",
    encoding="utf-8",
)

for row in read_sales_csv(messy_csv):
    print(row)

Look at row 206: `DictReader` did not crash on the extra field. It put the surplus
value in a list under the key `None`. Quiet surprises like that are why validation has
to be written, not assumed.

Session 4's Example 18 collected *every* problem with a row rather than stopping at the
first. Here is the same idea, adapted to rows read from a CSV:

In [ ]:
def validate_sale(row: dict) -> list[str]:
    """Return a list of problems with a raw CSV row. An empty list means the row is valid."""
    problems = []

    for field in sorted(REQUIRED_COLUMNS):
        if row.get(field) in ("", None):
            problems.append(f"empty {field!r}")

    if None in row:
        problems.append(f"unexpected extra fields {row[None]}")

    if row.get("price") and parse_price(row["price"]) is None:
        problems.append(f"unusable price {row['price']!r}")

    return problems


def clean_sales(rows: list[dict]) -> tuple[list[dict], list[tuple[str, list[str]]]]:
    """Split rows into (converted valid sales, rejected (id, problems) pairs)."""
    kept, rejected = [], []
    for row in rows:
        problems = validate_sale(row)
        if problems:
            rejected.append((row.get("id"), problems))
        else:
            kept.append(convert_sale(row))
    return kept, rejected


raw_rows = read_sales_csv(messy_csv)
clean_rows, rejected = clean_sales(raw_rows)

print(f"rows read    : {len(raw_rows)}")
print(f"rows kept    : {len(clean_rows)}")
print(f"rows rejected: {len(rejected)}")
for row_id, problems in rejected:
    print(f"  id {row_id}: {'; '.join(problems)}")

Two policies are visible in that output, and both are decisions somebody must own:

- `"$4.25"` was **repaired** (`parse_price` strips the `$`), not rejected.
- `"-2.75"` was **rejected**, although it may be a legitimate refund.

A project manager should be able to find those decisions in the code, and see their
effect in the counts the program reports.

### Write CSV output with `csv.DictWriter`

Now the full read → clean → analyze → write loop. The analysis uses Session 4's
`group_by`, and the result is written as a new file in `output/`. The raw input is
never modified.

In [ ]:
def group_by(rows: list[dict], column: str) -> dict[object, list[dict]]:
    """Return rows grouped into a dictionary keyed by one column's value."""
    groups: dict[object, list[dict]] = {}
    for row in rows:
        groups.setdefault(row[column], []).append(row)
    return groups


def item_revenue(rows: list[dict]) -> list[dict]:
    """Return one summary row per item: units sold and revenue, highest revenue first."""
    summary = []
    for item, item_rows in group_by(rows, "item").items():
        summary.append({
            "item": item,
            "units": len(item_rows),
            "revenue": round(sum(row["price"] for row in item_rows), 2),
        })
    return sorted(summary, key=lambda row: (-row["revenue"], row["item"]))


revenue_rows = item_revenue(converted)
revenue_csv = WORKSPACE / "output" / "item_revenue.csv"

with open(revenue_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["item", "units", "revenue"])
    writer.writeheader()
    writer.writerows(revenue_rows)

print(revenue_csv.read_text(encoding="utf-8"))

### Verify the generated CSV

Read the output back and compare it with what we meant to write. Remember that the
numbers return as strings, so convert before comparing:

In [ ]:
with open(revenue_csv, encoding="utf-8", newline="") as file:
    written_rows = list(csv.DictReader(file))

assert [row["item"] for row in written_rows] == [row["item"] for row in revenue_rows]
assert sum(int(row["units"]) for row in written_rows) == len(converted), "units do not add up"
assert round(sum(float(row["revenue"]) for row in written_rows), 2) == round(sum(s["price"] for s in sales), 2)

print(f"verified {len(written_rows)} rows; units and revenue reconcile with the input")

The last two checks are **reconciliation**: the units in the summary add up to the
number of transactions, and the revenue adds up to the total of the input. A summary
that does not reconcile with its input is wrong somewhere, even if every line of code
"worked".

In [ ]:
### Try it: write output/day_revenue.csv with columns day, sales, revenue,
### using group_by(converted, "day"). Keep the days in Mon..Fri order, not
### alphabetical order. Then read it back and check that revenue reconciles.

### Enter your code here ###

---

## 5. JSON files

**JSON** — JavaScript Object Notation — is language-independent despite its name. It
is the standard format for web APIs, configuration files, and experiment metadata,
because unlike CSV it can represent **nested** structure and a handful of **types**.

### JSON values and their Python equivalents

| JSON | Python | Example |
|---|---|---|
| object | `dict` | `{"item": "coffee", "price": 3.5}` |
| array | `list` | `["Mon", "Tue"]` |
| string | `str` | `"coffee"` |
| number | `int` or `float` | `101`, `3.5` |
| `true` / `false` | `True` / `False` | |
| `null` | `None` | |

`json.dumps` ("dump to **s**tring") shows the translation:

In [ ]:
import json

example = {"item": "coffee", "price": 3.5, "hours": [8, 9, 10], "seasonal": False, "supplier": None}
text = json.dumps(example)

print("Python:", example)
print("JSON  :", text)
print("type of the JSON:", type(text).__name__)

The JSON keywords are lowercase: `true`, `false`, `null`. Typing `True` or `None` into a
`.json` file by hand makes it invalid. JSON strings must use **double** quotes; Python
accepts either.

### A nested document: the cart configuration

CSV suits one flat table. The coffee cart's configuration is not a table: it has a
location, opening hours, a menu of items grouped by category, and a list of accepted
payment types. JSON represents that shape directly.

In [ ]:
cart_config = {
    "cart_id": "cart-01",
    "name": "Library Plaza Coffee Cart",
    "location": {"building": "Library South", "campus": "Downtown"},
    "open_hours": {"start": 8, "end": 12},
    "menu": {
        "drink": [{"item": "coffee", "price": 3.50}, {"item": "tea", "price": 2.75}],
        "food": [
            {"item": "muffin", "price": 2.50},
            {"item": "bagel", "price": 3.00},
            {"item": "cookie", "price": 1.75},
        ],
    },
    "payment_types": ["card", "campus_id"],
    "accepts_cash": False,
    "manager": "Zoë Café-Martín",
}

config_path = WORKSPACE / "data" / "raw" / "cart_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(cart_config, file, indent=2, ensure_ascii=False)

print(config_path.read_text(encoding="utf-8"))

- `indent=2` pretty-prints for human readers. Without it the whole document is written
  on one line — valid, but hard to review.
- `ensure_ascii=False` writes `Zoë` as `Zoë`. The default, `True`, writes `Zo\u00eb`:
  still correct JSON, but unreadable in a text editor. Pair it with `encoding="utf-8"`.

### Read JSON with `json.load`

`json.load(file)` reads the **entire** file and returns one Python value — here, a
dictionary. After that it is ordinary Python; nothing about it is "JSON" any more.

In [ ]:
with open(config_path, encoding="utf-8") as file:
    config = json.load(file)

print(type(config).__name__)
print(config["name"])
print(config["location"]["building"])            # dict inside dict
print(config["menu"]["food"][1]["item"])          # dict -> list -> dict
print(config == cart_config)

Read `config["menu"]["food"][1]["item"]` left to right, exactly as Session 3 taught you
to read `sales[0]["item"]`: each bracket peels off one layer. Walking a nested structure
is just nested loops:

In [ ]:
for category, items in config["menu"].items():
    names = ", ".join(f"{entry['item']} (${entry['price']:.2f})" for entry in items)
    print(f"{category:<5}: {names}")

### JSON round trips are not perfect either

JSON keeps more than CSV, but not everything. Session 3 counted sales by hour into a
dictionary with **integer** keys. JSON object keys must be strings, so watch what comes
back:

In [ ]:
from collections import Counter

sales_by_hour = dict(Counter(sale["hour"] for sale in sales))

print("before     :", sales_by_hour)

round_trip = json.loads(json.dumps(sales_by_hour))
print("after      :", round_trip)
print("key 8 there?", 8 in round_trip, "| key '8' there?", "8" in round_trip)

print("tuple      :", json.loads(json.dumps((8, 5))))

try:
    json.dumps({"days": {"Mon", "Tue"}})
except TypeError as error:
    print("set        :", f"TypeError: {error}")

| Python value | After a JSON round trip |
|---|---|
| `dict` with `int` keys | `dict` with `str` keys |
| `tuple` | `list` |
| `set` | cannot be written at all — convert with `sorted(...)` first |
| `float("nan")` | written as `NaN`, which is *not* valid standard JSON |

Convert deliberately on the way out (for example `sorted(days)`) and on the way back in
(`{int(hour): count for hour, count in data.items()}`).

### Validate the structure

`json.load` guarantees the file is valid JSON. It does not guarantee it is the JSON your
program expects. Check the shape before using it:

In [ ]:
def validate_config(config: object) -> list[str]:
    """Return a list of problems with a cart configuration. Empty means valid."""
    if not isinstance(config, dict):
        return [f"top level must be an object, got {type(config).__name__}"]

    problems = []
    for field in ("cart_id", "name", "menu"):
        if field not in config:
            problems.append(f"missing field {field!r}")

    menu = config.get("menu", {})
    if not isinstance(menu, dict):
        problems.append("menu must be an object of category -> list of items")
    else:
        for category, items in menu.items():
            for position, entry in enumerate(items):
                if not isinstance(entry.get("price"), (int, float)):
                    problems.append(f"menu.{category}[{position}] price must be a number")
    return problems


print("good config:", validate_config(config))
print("a list     :", validate_config([config]))
print("broken     :", validate_config({"cart_id": "x", "menu": {"drink": [{"item": "tea", "price": "2.75"}]}}))

### `load` / `loads` and `dump` / `dumps`

| Function | Works with | Does |
|---|---|---|
| `json.load(file)` | an open file | reads JSON text → Python value |
| `json.loads(text)` | a string | parses JSON text → Python value |
| `json.dump(value, file)` | an open file | writes Python value → JSON text |
| `json.dumps(value)` | — | returns Python value → JSON text as a `str` |

The trailing **s** stands for **string**. Use `load`/`dump` with files, and
`loads`/`dumps` when the JSON is already in a string — for example, one line of a JSONL
file, which is the next section.

### Invalid JSON

A parsing error reports the line and column where parsing failed, which is usually just
*after* the actual mistake:

In [ ]:
bad_texts = {
    "Python keyword": '{"accepts_cash": False}',
    "single quotes": "{'item': 'tea'}",
    "trailing comma": '{"item": "tea", "price": 2.75,}',
}

for label, bad in bad_texts.items():
    try:
        json.loads(bad)
    except json.JSONDecodeError as error:
        print(f"{label:<15} -> {error.msg} (line {error.lineno}, column {error.colno})")

In [ ]:
### Try it: add "espresso" at 2.25 to the drink menu in `config`, write it to
### data/processed/cart_config_v2.json, and read it back. Is config_path unchanged?
### Then save sales_by_hour to JSON and write the one line of code that restores
### integer keys when you load it.

### Enter your code here ###

---

## 6. JSON Lines (JSONL)

**JSON Lines** — also called newline-delimited JSON, or NDJSON — stores **one complete
JSON value per line**. The file as a whole is *not* a JSON document; each line is.

```text
{"id": 101, "item": "coffee", "price": 3.5, ...}
{"id": 102, "item": "tea", "price": 2.75, ...}
```

It exists because many datasets are a long sequence of independent records: log
entries, app events, API results, the prompts and responses of an LLM evaluation. JSONL
lets a program write each record as it happens, read one record at a time without
holding the whole file in memory, and point at the exact line that is broken.

### Write JSONL

One `json.dumps` per record, one `"\n"` after each. No `indent` — a record must stay on
a single line.

In [ ]:
sales_jsonl = WORKSPACE / "data" / "raw" / "sales.jsonl"

with open(sales_jsonl, "w", encoding="utf-8") as file:
    for sale in sales:
        file.write(json.dumps(sale, ensure_ascii=False) + "\n")

with open(sales_jsonl, encoding="utf-8") as file:
    for line in list(file)[:4]:
        print(line, end="")
print("...")

### `json.load` does not read JSONL

In [ ]:
try:
    with open(sales_jsonl, encoding="utf-8") as file:
        json.load(file)
except json.JSONDecodeError as error:
    print(f"JSONDecodeError: {error}")

*Extra data* means: the first line parsed as a complete value, and then there was more.
From `json.load`'s point of view that is an error; from JSONL's point of view it is the
whole design.

### Read JSONL

Loop over lines, skip blank ones, and `json.loads` each:

In [ ]:
records = []
with open(sales_jsonl, encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))

print(f"read {len(records)} records")
print("types preserved?", records == sales)
print("price type      :", type(records[0]["price"]).__name__)

Unlike the CSV round trip, numbers came back as numbers. JSONL keeps JSON's types and
nesting while keeping CSV's one-record-per-line convenience.

### Stream instead of collecting

The loop above builds a list, which holds every record in memory. For a large file, do
the work as each record arrives and keep only the running result — the Session 3
accumulator pattern, fed by a file:

In [ ]:
revenue_by_category = {}
record_count = 0

with open(sales_jsonl, encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue
        sale = json.loads(line)
        record_count += 1
        category = sale["category"]
        revenue_by_category[category] = revenue_by_category.get(category, 0) + sale["price"]

print(record_count, "records streamed")
print(revenue_by_category)

This loop uses the same memory whether the file holds twelve sales or twelve million.

### Append a record

Appending is where JSONL shines. Mode `"a"` adds a line; nothing already in the file has
to be read, parsed, or rewritten.

In [ ]:
new_sale = {"id": 113, "item": "tea", "category": "drink", "price": 2.75, "day": "Fri", "hour": 9, "customer_type": "faculty"}

with open(sales_jsonl, "a", encoding="utf-8") as file:
    file.write(json.dumps(new_sale) + "\n")

with open(sales_jsonl, encoding="utf-8") as file:
    lines = file.readlines()
print(len(lines), "lines; last one:", lines[-1], end="")

Doing the same with an ordinary JSON array means loading the whole array, appending in
memory, and rewriting the entire file — and if the program crashes half-way through the
rewrite, the file is left invalid.

### Report bad lines by number

A JSONL file can be damaged one line at a time: a truncated write, a hand edit, a
Python-style `None`. A program should say **which line**, because "invalid JSON" in a
file of 8 million lines is not an actionable message.

In [ ]:
orders_jsonl = WORKSPACE / "data" / "raw" / "mobile_orders.jsonl"
orders_jsonl.write_text(
    '{"order_id": "m001", "item": "coffee", "price": 3.50}\n'
    '{"order_id": "m002", "item": "tea", "price": 2.75}\n'
    "\n"
    '{"order_id": "m003", "item": "muffin", "price": None}\n'
    '{"order_id": "m004", "item": "bagel", "price": 3.00}\n'
    '{"order_id": "m005", "item": "cookie"\n'
    '{"order_id": "m006", "item": "coffee", "price": 3.50}\n',
    encoding="utf-8",
)


def read_jsonl_strict(path: Path) -> list[dict]:
    """Return every record, raising ValueError at the first invalid line."""
    records = []
    with open(path, encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"{path.name}, line {line_number}: {error.msg}") from error
    return records


try:
    read_jsonl_strict(orders_jsonl)
except ValueError as error:
    print("ValueError:", error)

`raise ... from error` keeps the original `JSONDecodeError` attached, so the full detail
is still there for whoever debugs it, while the message on top is written for whoever
has to fix the file.

**Stop or skip?** Stopping at the first bad line is right when a partial answer would be
misleading — a financial total, say. For other jobs, skipping and **counting** the bad
lines is better, as long as the count is reported. It is Session 4's `parse_price` policy
again, applied to whole records:

In [ ]:
def read_jsonl_lenient(path: Path) -> tuple[list[dict], list[tuple[int, str]]]:
    """Return (records, errors), where errors lists (line number, message) for skipped lines."""
    records, errors = [], []
    with open(path, encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                errors.append((line_number, error.msg))
    return records, errors


orders, errors = read_jsonl_lenient(orders_jsonl)
print(f"read {len(orders)} orders, skipped {len(errors)} lines")
for line_number, message in errors:
    print(f"  line {line_number}: {message}")
print("order ids:", [order["order_id"] for order in orders])

Notice that the blank line 3 was skipped silently while lines 4 and 6 were reported. That,
too, is a policy: a blank line is harmless, a broken record is lost data.

### JSON versus JSONL

| | JSON | JSONL |
|---|---|---|
| Structure | one value for the whole file | one value per line |
| Read with | `json.load(file)` | a loop of `json.loads(line)` |
| Write with | `json.dump(value, file)` | a loop of `file.write(json.dumps(record) + "\n")` |
| Append a record | rewrite the whole file | open with `"a"`, write one line |
| Memory for a big file | the whole document | one record at a time |
| One corrupt spot | whole file unreadable | one line lost, located by number |
| Best for | one coherent document: config, metadata | many independent records: logs, events, results |

A JSON file answers *"what is this document?"*; a JSONL file answers *"what are the
records in this collection?"*. And an extension is only a convention — a file called
`.json` can contain JSONL, or garbage. Validate the content, not the name.

In [ ]:
### Try it: append a line containing just   {"order_id": "m007"   (no closing brace)
### to mobile_orders.jsonl. Predict which line number read_jsonl_lenient reports.
### Then write a function that also rejects valid JSON records lacking a "price"
### number, and reports them separately from the parse errors.

### Enter your code here ###

---

## 7. Choosing CSV, JSON, or JSONL

The same twelve sales can be stored in all three formats. Compare them:

In [ ]:
sales_json = WORKSPACE / "data" / "processed" / "sales.json"
with open(sales_json, "w", encoding="utf-8") as file:
    json.dump(sales, file, indent=2)

sales_jsonl_clean = WORKSPACE / "data" / "processed" / "sales.jsonl"
with open(sales_jsonl_clean, "w", encoding="utf-8") as file:
    for sale in sales:
        file.write(json.dumps(sale) + "\n")

for path in (sales_csv, sales_json, sales_jsonl_clean):
    text = path.read_text(encoding="utf-8")
    print(f"{path.name:<12} {len(text):>5} characters {len(text.splitlines()):>4} lines")

CSV is the most compact because it names each column once, in the header. JSON and JSONL
repeat every key on every record — the price of carrying types and structure. For a flat
table like this, that is a real cost; for nested records, it is the only way to keep the
structure.

| | CSV | JSON | JSONL |
|---|---|---|---|
| Shape | flat rows and columns | nested document | one record per line |
| Types | none — all text | strings, numbers, booleans, null | same as JSON |
| Nested data | must be flattened | natural | natural, per record |
| Python reader | `csv.DictReader` | `json.load` | `json.loads` per line |
| Opens in a spreadsheet | yes | no | no |
| Streams a large file | yes | awkwardly | yes |
| Appending records | yes, if the columns match | awkward | trivial |
| Typical use | exports, reports, simple tables | APIs, configs, metadata | logs, events, model outputs |

### Flattening: nested data into a table

When nested data has to go to a spreadsheet user, you flatten it: each nested path becomes
a column name. Here is the cart menu flattened into CSV rows:

In [ ]:
flat_menu = [
    {"cart_id": config["cart_id"], "category": category, "item": entry["item"], "price": entry["price"]}
    for category, entries in config["menu"].items()
    for entry in entries
]

flat_menu_csv = WORKSPACE / "output" / "menu_flat.csv"
with open(flat_menu_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["cart_id", "category", "item", "price"])
    writer.writeheader()
    writer.writerows(flat_menu)

print(flat_menu_csv.read_text(encoding="utf-8"))

The menu flattened cleanly because each item has the same fields. The configuration as a
*whole* — location, opening hours, payment types — does not fit into this table at all.
That is the clearest sign a dataset belongs in JSON rather than CSV.

---

## 8. JSON data-quality concerns

JSON avoids CSV's quoting and type problems, and introduces problems of its own.

### Schema inconsistency

Nothing forces every record to have the same keys:

In [ ]:
student_records = [
    {"student_id": "1001", "name": "Amina Patel", "program": "MS Data Science", "gpa": 3.8},
    {"student_id": "1002", "name": "Jordan Lee", "program": "MS Analytics"},
    {"student_id": "1003", "name": "Carlos Rivera", "program": "MS Data Science", "gpa": "3.9"},
    {"student_id": "1004", "name": "Mei Chen", "program": "MS Analytics", "gpa": None},
    {"student_id": "1005", "name": "Sam Okafor", "program": "MS Data Science", "gpa": "not available"},
]

for student in student_records:
    try:
        print(student["name"], "->", student["gpa"])
    except KeyError as error:
        print(student["name"], "-> KeyError:", error)

`student["gpa"]` raises `KeyError` when the key is absent. `student.get("gpa")` returns
`None` instead. Which one to use is a question about the data, not about Python:

- If the field is **optional**, `.get()` is right.
- If the field is **required**, a missing key is an error in the input, and the program
  should say so clearly rather than carry on with `None`.

### Type inconsistency

The five records hold GPA as a number, missing, a string of a number, `null`, and a
string that is not a number. They look related; they mean different things. A program
must classify them explicitly:

In [ ]:
def classify_gpa(record: dict) -> tuple[str, float | None]:
    """Return (status, gpa) where status explains how the value was treated."""
    if "gpa" not in record:
        return "missing key", None
    value = record["gpa"]
    if value is None:
        return "null", None
    if isinstance(value, bool):
        return "invalid type", None
    if isinstance(value, (int, float)):
        return "number", float(value)
    if isinstance(value, str):
        try:
            return "numeric string", float(value)
        except ValueError:
            return "unparseable string", None
    return "invalid type", None


status_counts = Counter()
for student in student_records:
    status, gpa = classify_gpa(student)
    status_counts[status] += 1
    print(f"{student['name']:<14} {status:<19} {gpa}")

print()
print(dict(status_counts))

Session 4's `safe_divide` needed an `isinstance(value, bool)` guard, and so does this:
`True` is an `int` in Python, so without the guard a GPA of `true` would quietly become
`1.0`.

### Nested data and large arrays

Nested JSON represents real structure faithfully, but analysis tools usually want tables.
A team has to decide, per nested field, whether to keep it as metadata, flatten it into
columns, split it into a separate related table, or leave it out.

And a JSON file that is one enormous array has to be loaded whole by `json.load`. When the
records are independent and the file is large, JSONL is usually the better operational
format.

---

## 9. Complete example: JSONL events in, CSV summary out

This program pulls the whole session together. The coffee cart's mobile-ordering app logs
one event per line: an order is placed, becomes ready, is picked up, or is cancelled. The
manager wants a table of how often each event type occurs.

```text
data/raw/app_events.jsonl ──▶ read → validate → count → write ──▶ output/event_summary.csv
```

First, the input file:

In [ ]:
events_jsonl = WORKSPACE / "data" / "raw" / "app_events.jsonl"

events = [
    {"event_id": "e001", "order_id": "m001", "event_type": "order_placed", "hour": 8},
    {"event_id": "e002", "order_id": "m001", "event_type": "order_ready", "hour": 8},
    {"event_id": "e003", "order_id": "m002", "event_type": "order_placed", "hour": 8},
    {"event_id": "e004", "order_id": "m001", "event_type": "picked_up", "hour": 8},
    {"event_id": "e005", "order_id": "m002", "event_type": "order_cancelled", "hour": 9},
    {"event_id": "e006", "order_id": "m003", "event_type": "order_placed", "hour": 9},
    {"event_id": "e007", "order_id": "m003", "event_type": "order_ready", "hour": 9},
    {"event_id": "e008", "order_id": "m003", "event_type": "picked_up", "hour": 10},
]

with open(events_jsonl, "w", encoding="utf-8") as file:
    for event in events:
        file.write(json.dumps(event) + "\n")

print(events_jsonl.read_text(encoding="utf-8"))

Now the program. Read it as four functions with one job each, and a `main()` that shows
the whole pipeline in five lines. Every function is one you have seen a version of above.

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

REQUIRED_EVENT_FIELDS = {"event_id", "event_type"}


def read_events(input_path: Path) -> list[dict]:
    """Read event records from a JSONL file.

    Raises:
        FileNotFoundError: if the file does not exist.
        ValueError: on invalid JSON or a record missing a required field,
            naming the line number.
    """
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path.resolve()}")

    events = []
    with open(input_path, mode="r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON on line {line_number}: {error.msg}") from error

            missing = REQUIRED_EVENT_FIELDS - set(event)
            if missing:
                raise ValueError(f"Line {line_number} is missing fields: {sorted(missing)}")

            events.append(event)

    return events


def count_event_types(events: list[dict]) -> Counter:
    """Count events by event type."""
    return Counter(event["event_type"] for event in events)


def write_event_summary(event_counts: Counter, output_path: Path) -> None:
    """Write event counts as a CSV file, one row per event type, sorted by type."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, mode="w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["event_type", "event_count"])
        writer.writeheader()
        for event_type, count in sorted(event_counts.items()):
            writer.writerow({"event_type": event_type, "event_count": count})


def main(input_path: Path, output_path: Path) -> None:
    events = read_events(input_path)
    event_counts = count_event_types(events)
    write_event_summary(event_counts, output_path)

    print(f"Input file : {input_path}")
    print(f"Events read: {len(events)}")
    print(f"Output file: {output_path}")
    for event_type, count in sorted(event_counts.items()):
        print(f"  {event_type:<16} {count}")


main(events_jsonl, WORKSPACE / "output" / "event_summary.csv")

Verify the output by reading it back:

In [ ]:
summary_csv = WORKSPACE / "output" / "event_summary.csv"
print(summary_csv.read_text(encoding="utf-8"))

with open(summary_csv, encoding="utf-8", newline="") as file:
    total = sum(int(row["event_count"]) for row in csv.DictReader(file))
assert total == len(events), "summary does not reconcile with the input"
print("reconciled:", total, "events in, ", total, "events counted")

And confirm the failure modes produce messages a person can act on:

In [ ]:
broken_events = WORKSPACE / "data" / "raw" / "app_events_broken.jsonl"
broken_events.write_text(
    '{"event_id": "e001", "event_type": "order_placed"}\n'
    '{"event_id": "e002"}\n',
    encoding="utf-8",
)

for path in (WORKSPACE / "data" / "raw" / "no_such_file.jsonl", broken_events, orders_jsonl):
    try:
        main(path, WORKSPACE / "output" / "should_not_exist.csv")
    except (FileNotFoundError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

print("partial output written?", (WORKSPACE / "output" / "should_not_exist.csv").exists())

The last line matters. Because `read_events` validates the **whole** input before
`write_event_summary` runs, a bad input produces no output at all — rather than a
half-written summary that someone might mistake for a real one.

This example illustrates every habit from the session:

- explicit input and output paths, relative to one workspace
- record-by-record reading of JSONL, with line numbers in every error
- validation before any output is written
- separate functions for reading, analysis, and writing
- a printed summary of what happened
- verification by reading the output back and reconciling it with the input

It still lives inside a notebook, though, and its `main()` is called by a notebook cell.
The companion notebook, **`Python_Modules_and_Scripts_orig.ipynb`**, takes the next step:
putting code like this into `.py` files that can be imported, tested, and run from a
terminal.

---

## 10. In-class activity

Work in pairs. Each task uses files already in `files_workspace/`.

**Task A — hourly report.** Read `data/raw/sales.jsonl` (it has 13 records now: you
appended one). Stream it, count sales by hour, and write `output/hourly_sales.csv` with
columns `hour,sales,revenue`, sorted by hour **numerically**. Read it back and check that
the `sales` column adds up to the number of records.

**Task B — clean the messy CSV.** Using `read_sales_csv` and `clean_sales`, write the
*kept* rows of `data/raw/sales_messy.csv` to `data/processed/sales_clean.csv` and the
*rejected* ids with their problems to `output/rejected_rows.jsonl`, one JSON object per
rejected row. Why is JSONL a better fit than CSV for the rejects? (Hint: how many problems
can one row have?)

**Task C — the manager's questions.** For Task B, write down in a markdown cell the answers
to: what is the input, who would provide it, which rows are dropped and why, and how would
someone check that the output is complete?

In [ ]:
### Task A

### Enter your code here ###

In [ ]:
### Task B

### Enter your code here ###

### Task C

*Enter your answers here.*

---

## Summary

| Topic | Remember |
|---|---|
| Working directory | Relative paths resolve from `Path.cwd()`; in Jupyter, that is the notebook's folder. |
| `pathlib` | Build paths with `/`; print `path.resolve()` when a file cannot be found. |
| `with open(...)` | Closes the file however the block ends. Always pass `encoding="utf-8"` for text. |
| Modes | `"r"` read, `"w"` **replace**, `"a"` append, `"x"` create-or-fail. |
| Lines | Iterate over the file; strip deliberately; `next(file, None)` for a header. |
| CSV | Use `csv`, never `split(",")`. `newline=""`. `DictReader`/`DictWriter`. Every value comes back as text. |
| JSON | `load`/`dump` for files, `loads`/`dumps` for strings. Nesting and basic types survive; `int` keys, tuples, and sets do not. |
| JSONL | One JSON value per line. Stream it, append to it, and report bad lines by number. |
| Validation | Check the file, the columns or fields, and the types **before** computing or writing anything. |
| Verification | Read the output back and reconcile it with the input. |

### Key takeaways

- Files give data and results a life beyond one notebook session, which is what makes a
  workflow repeatable.
- Most early `FileNotFoundError`s are working-directory problems.
- Choose the format by the data's shape: CSV for flat tables, JSON for one structured
  document, JSONL for many independent records.
- Formats lose information in round trips — types in CSV, key types in JSON — so convert
  explicitly right after reading.
- Keep raw inputs separate from generated outputs, and never write over the raw data.
- Validate early, report counts of what was read, kept, and rejected, and verify outputs
  by reading them back.

### Where to go next

- **`Python_Modules_and_Scripts_orig.ipynb`** — move this code into modules and a
  runnable script.
- Python documentation: [Reading and writing files](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files),
  [`pathlib`](https://docs.python.org/3/library/pathlib.html),
  [`csv`](https://docs.python.org/3/library/csv.html),
  [`json`](https://docs.python.org/3/library/json.html), and the
  [JSON Lines format](https://jsonlines.org/).